# Customer Data Cleaning & Preprocessing with Python & Pandas

**Auteur :** [Ton nom]
**Date :** [Date]
**Contexte :** Projet de portfolio — simulation d'une mission client de nettoyage de données.

---

## 🎯 Objectif

Nettoyer et préparer un dataset client brut contenant volontairement des anomalies
(doublons, valeurs aberrantes, formats incohérents, emails invalides, valeurs manquantes)
afin de produire un dataset propre, exploitable et prêt pour l'analyse.

**Pipeline :** Raw Dataset → Inspection → Détection des problèmes → Nettoyage → Validation → Clean Dataset → Export

## 📑 Sommaire

1. Introduction
2. Dataset
3. Initial Data Inspection
4. Data Quality Issues
5. Data Cleaning
6. Validation
7. Before vs After
8. Final Dataset
9. Conclusion

In [173]:
# ============================================================
# 1. IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Pandas version :", pd.__version__)

Pandas version : 2.3.3


---
## 1. Introduction

Ce notebook présente un pipeline complet de **nettoyage de données clients**
réalisé avec **Python et Pandas**.

Le dataset brut a été volontairement construit avec des imperfections réalistes :

- doublons,
- valeurs d'âge aberrantes (négatives, > 100),
- noms et villes mal formatés (espaces, majuscules incohérentes),
- emails invalides,
- valeurs manquantes.

L'objectif est de démontrer une démarche professionnelle :
**inspecter → diagnostiquer → nettoyer → valider → exporter**.

---
## 2. Dataset

Le dataset est chargé depuis `data/raw/customers_raw.csv`.

In [174]:
# ============================================================
# 2. CHARGEMENT DU DATASET BRUT
# ============================================================
df = pd.read_csv("data/raw/customers_raw.csv")

# Sauvegarde d'une copie brute AVANT tout nettoyage
df_raw = df.copy()

df.head()

,customer_id,name,age,email,city,purchase_amount
0,101,john smith,25.0,john.smith@gmail.com,agadir,250.50
1,102,Marie DUPONT,31.0,marie.dupont@gmail.com,AGADIR,120.00
2,103,paul martin,150.0,paul.martin@gmail.com,Casablanca,300.75
3,104,Sarah johnson,28.0,sarah.johnson@gmail.com,casablanca,NaN
4,105,NaN,NaN,invalid-email,Rabat,90.00


### 📊 Aperçu du dataset brut

- **8 lignes** (7 clients uniques + 1 doublon)
- **6 colonnes** : `customer_id`, `name`, `age`, `email`, `city`, `purchase_amount`

In [175]:
print("Dimensions :", df.shape)
print("\nColonnes :", list(df.columns))
df

Dimensions : (8, 6)

Colonnes : ['customer_id', 'name', 'age', 'email', 'city', 'purchase_amount']


,customer_id,name,age,email,city,purchase_amount
0,101,john smith,25.0,john.smith@gmail.com,agadir,250.50
1,102,Marie DUPONT,31.0,marie.dupont@gmail.com,AGADIR,120.00
2,103,paul martin,150.0,paul.martin@gmail.com,Casablanca,300.75
3,104,Sarah johnson,28.0,sarah.johnson@gmail.com,casablanca,NaN
4,105,NaN,NaN,invalid-email,Rabat,90.00
5,106,mohamed ali,22.0,mohamed.ali@gmail.com,RABAT,450.00
6,107,Emma Brown,-5.0,emma.brown@gmail.com,agadir,200.00
7,107,Emma Brown,-5.0,emma.brown@gmail.com,agadir,200.00


---
## 3. Initial Data Inspection

Première inspection : types, valeurs manquantes, doublons, statistiques.

In [176]:
# ============================================================
# 3. INSPECTION INITIALE
# ============================================================
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      8 non-null      int64  
 1   name             7 non-null      object 
 2   age              7 non-null      float64
 3   email            8 non-null      object 
 4   city             8 non-null      object 
 5   purchase_amount  7 non-null      float64
dtypes: float64(2), int64(1), object(3)
memory usage: 516.0+ bytes


In [177]:
# Valeurs manquantes par colonne
print("Valeurs manquantes :")
print(df.isnull().sum())
print("\nTotal :", df.isnull().sum().sum())

Valeurs manquantes :
customer_id        0
name               1
age                1
email              0
city               0
purchase_amount    1
dtype: int64

Total : 3


In [178]:
# Doublons complets
print("Doublons complets :", df.duplicated().sum())

# Doublons sur customer_id (clé unique attendue)
print("\nDoublons sur customer_id :")
df[df["customer_id"].duplicated(keep=False)]

Doublons complets : 1

Doublons sur customer_id :


,customer_id,name,age,email,city,purchase_amount
6,107,Emma Brown,-5.0,emma.brown@gmail.com,agadir,200.0
7,107,Emma Brown,-5.0,emma.brown@gmail.com,agadir,200.0


In [179]:
df.describe(include="all")

,customer_id,name,age,email,city,purchase_amount
count,8.000000,7,7.000000,8,8,7.000000
unique,NaN,6,NaN,7,6,NaN
top,NaN,Emma Brown,NaN,emma.brown@gmail.com,agadir,NaN
freq,NaN,2,NaN,2,3,NaN
mean,104.375000,NaN,35.142857,NaN,NaN,230.178571
std,2.263846,NaN,52.850193,NaN,NaN,120.641058
min,101.000000,NaN,-5.000000,NaN,NaN,90.000000
25%,102.750000,NaN,8.500000,NaN,NaN,160.000000
50%,104.500000,NaN,25.000000,NaN,NaN,200.000000
75%,106.250000,NaN,29.500000,NaN,NaN,275.625000


---
## 4. Data Quality Issues

### 🚨 Problèmes détectés

| # | Problème | Colonne | Détail |
|---|----------|---------|--------|
| 1 | Doublon complet | toutes | `customer_id = 107` (Emma Brown) apparaît 2 fois |
| 2 | Âges invalides | `age` | 2 valeurs `-5` et 1 valeur `150` |
| 3 | Valeur manquante | `age` | 1 `NaN` |
| 4 | Valeur manquante | `name` | 1 `None` |
| 5 | Valeur manquante | `purchase_amount` | 1 `NaN` |
| 6 | Noms mal formatés | `name` | espaces + casse incohérente (`"  john smith"`, `"Marie DUPONT "`) |
| 7 | Villes mal formatées | `city` | casse + espaces (`"agadir"`, `"AGADIR"`, `" casablanca "`) |
| 8 | Email invalide | `email` | `"invalid-email"` ne respecte pas le format |

In [180]:
# Diagnostic des âges
print("Âges triés :")
print(df["age"].sort_values().tolist())

print("\nÂges aberrants (< 0 ou > 100) :")
df[(df["age"] < 0) | (df["age"] > 100)]

Âges triés :
[-5.0, -5.0, 22.0, 25.0, 28.0, 31.0, 150.0, nan]

Âges aberrants (< 0 ou > 100) :


,customer_id,name,age,email,city,purchase_amount
2,103,paul martin,150.0,paul.martin@gmail.com,Casablanca,300.75
6,107,Emma Brown,-5.0,emma.brown@gmail.com,agadir,200.00
7,107,Emma Brown,-5.0,emma.brown@gmail.com,agadir,200.00


In [181]:
# Diagnostic des villes
print("Villes uniques (brut) :")
print(df["city"].value_counts())

Villes uniques (brut) :
city
agadir          3
AGADIR          1
Casablanca      1
 casablanca     1
Rabat           1
RABAT           1
Name: count, dtype: int64


In [182]:
# Diagnostic des emails
print("Emails :")
for e in df["email"].tolist():
    print(" -", e)

Emails :
 - john.smith@gmail.com
 - marie.dupont@gmail.com
 - paul.martin@gmail.com
 - sarah.johnson@gmail.com
 - invalid-email
 - mohamed.ali@gmail.com
 - emma.brown@gmail.com
 - emma.brown@gmail.com


---
## 5. Data Cleaning

Nous allons corriger chaque problème identifié, étape par étape.

### 5.1 — Suppression du doublon

Le `customer_id = 107` apparaît 2 fois avec des données strictement identiques.
On conserve une seule occurrence.

In [193]:
# Vérification avant suppression
print("Avant :", df.shape)

df = df.drop_duplicates().copy()

print("Après :", df.shape)
print("Doublons restants :", df.duplicated().sum())

Avant : (7, 6)
Après : (7, 6)
Doublons restants : 0


### 5.2 — Standardisation des noms

On supprime les espaces superflus et on applique un format `Prénom Nom`.

In [194]:
df["name"] = df["name"].str.strip().str.title()

df[["customer_id", "name"]]

,customer_id,name
0,101,John Smith
1,102,Marie Dupont
2,103,Paul Martin
3,104,Sarah Johnson
4,105,NaN
5,106,Mohamed Ali
6,107,Emma Brown


### 5.3 — Standardisation des villes

Même logique : suppression des espaces + format `Title Case`.

In [195]:
df["city"] = df["city"].str.strip().str.title()

print(df["city"].value_counts())

city
Agadir        3
Casablanca    2
Rabat         2
Name: count, dtype: int64


### 5.4 — Nettoyage des âges

Stratégie :
1. Les âges invalides (< 0 ou > 100) sont marqués comme manquants (`pd.NA`).
2. Les valeurs manquantes sont imputées par la **médiane** (plus robuste que la moyenne).

In [196]:
# Marquer les âges invalides comme manquants
df.loc[(df["age"] < 0) | (df["age"] > 100), "age"] = pd.NA

# Calcul de la médiane (sur les valeurs valides)
median_age = df["age"].median()
print("Médiane des âges :", median_age)

# Imputation
df["age"] = df["age"].fillna(median_age)

# Conversion en entier (un âge est un entier)
df["age"] = df["age"].astype(int)

print("\nÂges après nettoyage :")
print(df["age"].sort_values().tolist())

Médiane des âges : 26.0

Âges après nettoyage :
[22, 25, 26, 26, 26, 28, 31]


### 5.5 — Validation des emails

On utilise une **expression régulière (regex)** pour vérifier le format `xxx@yyy.zzz`.
Les emails invalides sont neutralisés (`pd.NA`) plutôt que supprimés,
afin de conserver la ligne client.

In [197]:
# Pattern email
email_pattern = r"^[\w\.-]+@[\w\.-]+\.\w+$"

# Création d'une colonne temporaire de validation
df["email_valid"] = df["email"].str.match(email_pattern)

# Aperçu
print(df[["email", "email_valid"]])

                     email email_valid
0     john.smith@gmail.com        True
1   marie.dupont@gmail.com        True
2    paul.martin@gmail.com        True
3  sarah.johnson@gmail.com        True
4                     <NA>        <NA>
5    mohamed.ali@gmail.com        True
6     emma.brown@gmail.com        True


In [188]:
# Emails invalides → NA
df.loc[df["email_valid"] == False, "email"] = pd.NA

# Suppression de la colonne temporaire
df = df.drop("email_valid", axis=1)

print("Emails après nettoyage :")
print(df["email"].tolist())

Emails après nettoyage :
['john.smith@gmail.com', 'marie.dupont@gmail.com', 'paul.martin@gmail.com', 'sarah.johnson@gmail.com', <NA>, 'mohamed.ali@gmail.com', 'emma.brown@gmail.com']


### 5.6 — Traitement de `purchase_amount`

Une valeur manquante (`Sarah Johnson`) est imputée par la **médiane**.
> 💡 Dans un vrai contexte client, ce choix serait à valider : s'agit-il d'un achat non enregistré ?

In [189]:
median_purchase = df["purchase_amount"].median()
print("Médiane purchase_amount :", median_purchase)

df["purchase_amount"] = df["purchase_amount"].fillna(median_purchase)

print("\nValeurs manquantes restantes :")
print(df["purchase_amount"].isnull().sum())

Médiane purchase_amount : 225.25

Valeurs manquantes restantes :
0


---
## 6. Validation

Contrôle qualité complet après nettoyage.

In [190]:
print("=" * 50)
print("VALIDATION FINALE")
print("=" * 50)

print("\n📐 Dimensions :", df.shape)

print("\n📋 Types :")
print(df.dtypes)

print("\n❓ Valeurs manquantes :")
print(df.isnull().sum())

print("\n🔁 Doublons :", df.duplicated().sum())

print("\n📊 Statistiques :")
print(df.describe())

print("\n👀 Aperçu :")
df

VALIDATION FINALE

📐 Dimensions : (7, 6)

📋 Types :
customer_id          int64
name                object
age                  int64
email               object
city                object
purchase_amount    float64
dtype: object

❓ Valeurs manquantes :
customer_id        0
name               1
age                0
email              1
city               0
purchase_amount    0
dtype: int64

🔁 Doublons : 0

📊 Statistiques :
       customer_id        age  purchase_amount
count     7.000000   7.000000         7.000000
mean    104.000000  26.285714       233.785714
std       2.160247   2.751623       119.963921
min     101.000000  22.000000        90.000000
25%     102.500000  25.500000       160.000000
50%     104.000000  26.000000       225.250000
75%     105.500000  27.000000       275.625000
max     107.000000  31.000000       450.000000

👀 Aperçu :


,customer_id,name,age,email,city,purchase_amount
0,101,John Smith,25,john.smith@gmail.com,Agadir,250.50
1,102,Marie Dupont,31,marie.dupont@gmail.com,Agadir,120.00
2,103,Paul Martin,26,paul.martin@gmail.com,Casablanca,300.75
3,104,Sarah Johnson,28,sarah.johnson@gmail.com,Casablanca,225.25
4,105,NaN,26,<NA>,Rabat,90.00
5,106,Mohamed Ali,22,mohamed.ali@gmail.com,Rabat,450.00
6,107,Emma Brown,26,emma.brown@gmail.com,Agadir,200.00


---
## 7. Before vs After

Comparaison quantitative entre le dataset brut et le dataset nettoyé.

In [191]:
comparison = pd.DataFrame({
    "Avant": [
        df_raw.shape[0],
        df_raw.shape[1],
        df_raw.isnull().sum().sum(),
        df_raw.duplicated().sum(),
        df_raw["age"].min(),
        df_raw["age"].max(),
    ],
    "Après": [
        df.shape[0],
        df.shape[1],
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        df["age"].min(),
        df["age"].max(),
    ]
}, index=[
    "Lignes",
    "Colonnes",
    "Valeurs manquantes",
    "Doublons",
    "Âge min",
    "Âge max",
])

comparison

,Avant,Après
Lignes,8.0,7
Colonnes,6.0,6
Valeurs manquantes,3.0,2
Doublons,1.0,0
Âge min,-5.0,22
Âge max,150.0,31


---
## 8. Final Dataset

Export du dataset nettoyé vers `data/cleaned/customers_cleaned.csv`.

In [192]:
df.to_csv("data/cleaned/customers_cleaned.csv", index=False)
print("✅ Dataset exporté : data/cleaned/customers_cleaned.csv")

# Vérification par relecture
df_check = pd.read_csv("data/cleaned/customers_cleaned.csv")
df_check

✅ Dataset exporté : data/cleaned/customers_cleaned.csv


,customer_id,name,age,email,city,purchase_amount
0,101,John Smith,25,john.smith@gmail.com,Agadir,250.50
1,102,Marie Dupont,31,marie.dupont@gmail.com,Agadir,120.00
2,103,Paul Martin,26,paul.martin@gmail.com,Casablanca,300.75
3,104,Sarah Johnson,28,sarah.johnson@gmail.com,Casablanca,225.25
4,105,NaN,26,NaN,Rabat,90.00
5,106,Mohamed Ali,22,mohamed.ali@gmail.com,Rabat,450.00
6,107,Emma Brown,26,emma.brown@gmail.com,Agadir,200.00


---
## 9. Conclusion

### ✅ Résumé des actions

| Action | Détail |
|--------|--------|
| Doublon supprimé | `customer_id = 107` (Emma Brown) |
| Noms standardisés | `strip()` + `title()` |
| Villes standardisées | `strip()` + `title()` |
| Âges aberrants corrigés | `-5`, `-5`, `150` → `NA` puis imputés à la médiane |
| Âge converti en entier | `float64` → `int64` |
| Emails validés | regex, 1 email invalide neutralisé |
| `purchase_amount` imputé | 1 valeur manquante → médiane |

### 📈 Résultat

- **8 → 7 lignes** (doublon supprimé)
- **0 doublon**
- **Âges cohérents** : 22 à 31 ans
- **Formats homogènes** sur `name` et `city`
- **Dataset exporté** : `customers_cleaned.csv`

### 🔮 Pistes d'amélioration

- Validation plus poussée des emails (via `email-validator`).
- Détection d'outliers sur `purchase_amount` (méthode IQR ou Z-score).
- Ajout d'une colonne `age_group` (segmentation).
- Automatisation du pipeline via un script `clean.py` réutilisable.